In [9]:
globals().clear()

In [10]:
from pathlib import Path
import cv2
import numpy as np

IMG_SIZE = 128
BASE_DIR = Path("../data/kaggle_3m")

mask_paths = list(BASE_DIR.rglob("*mask*.*"))
image_paths = [Path(str(m).replace("_mask", "")) for m in mask_paths]

print(f" {len(mask_paths)} pairs of picture-mask")

def process_image(path):
    img = cv2.imread(str(path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img, (IMG_SIZE, IMG_SIZE)) / 255.0

def process_mask(path):
    mask = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return cv2.resize(mask, (IMG_SIZE, IMG_SIZE)) / 255.0

#CNN-friendly dims
X = np.array([process_image(p) for p in image_paths], dtype=np.float32)
Y = np.array([process_mask(p) for p in mask_paths], dtype=np.float32)

Y = Y[..., np.newaxis]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

 3929 pairs of picture-mask
X shape: (3929, 128, 128, 3)
Y shape: (3929, 128, 128, 1)


In [11]:
has_tumor = np.sum(Y, axis=(1, 2, 3)) > 0

total_images = len(Y)
positive_cases = np.sum(has_tumor)
negative_cases = total_images - positive_cases

print(f"Total: {total_images}")
print(f"Cancer: {positive_cases} ({positive_cases/total_images*100:.2f}%)")
print(f"No Cancer: {negative_cases} ({negative_cases/total_images*100:.2f}%)")

Total: 3929
Cancer: 1373 (34.95%)
No Cancer: 2556 (65.05%)


In [12]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import keras
model=keras.Sequential()
model.add(keras.Input(shape=(128,128,3)))
model.add(keras.layers.Conv2D(filters=32,
                              kernel_size=(3,3)))
model.add(keras.layers.MaxPool2D(pool_size=(2,2)))
model.add(keras.layers.Flatten())
model.add(keras.layers.Dense(1,activation='sigmoid'))

model.compile(optimizer=keras.optimizers.Adam(),
              loss=keras.losses.binary_crossentropy,
              metrics=['accuracy'])

In [13]:
Y_class = (np.sum(Y, axis=(1, 2, 3)) > 0).astype(np.float32)

print("New shape Y_class:", Y_class.shape) 

New shape Y_class: (3929,)


In [15]:
history = model.fit(X,Y_class,batch_size=32,epochs=5, verbose=2)

Epoch 1/5
123/123 - 2s - 13ms/step - accuracy: 0.8972 - loss: 0.2494
Epoch 2/5
123/123 - 2s - 13ms/step - accuracy: 0.9163 - loss: 0.2169
Epoch 3/5
123/123 - 2s - 13ms/step - accuracy: 0.9326 - loss: 0.1796
Epoch 4/5
123/123 - 2s - 13ms/step - accuracy: 0.9410 - loss: 0.1640
Epoch 5/5
123/123 - 2s - 13ms/step - accuracy: 0.9509 - loss: 0.1409
